# SLM — Exploration Notebook

Three sections:
1. **Dataset Exploration** — token distribution, sequence lengths, data quality checks
2. **Training Run** — data loading, tokenization, training loop, loss curves
3. **Inference** — load a checkpoint, generate text, compare outputs

---

In [ ]:
# Core imports used across all sections
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports ready.')

---
## Section 1 — Dataset Exploration

Inspect the curated dataset before training. Checks token distribution, sequence length distribution, and flags quality issues.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
DATA_PATH   = Path('../curator/output')   # path to curated jsonl files
TOKENIZER   = 'gpt2'                       # tokenizer to use for length analysis
SAMPLE_SIZE = 10_000                       # number of samples to inspect

print(f'Data path  : {DATA_PATH}')
print(f'Tokenizer  : {TOKENIZER}')
print(f'Sample size: {SAMPLE_SIZE:,}')

In [ ]:
# ── Load a sample of the curated dataset ──────────────────────────────────
records = []
for f in sorted(DATA_PATH.glob('*.jsonl')):
    with open(f) as fh:
        for line in fh:
            records.append(json.loads(line))
            if len(records) >= SAMPLE_SIZE:
                break
    if len(records) >= SAMPLE_SIZE:
        break

df = pd.DataFrame(records)
print(f'Loaded {len(df):,} records')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Character-level length distribution ───────────────────────────────────
text_col = 'text'   # update if your jsonl uses a different key

df['char_len'] = df[text_col].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['char_len'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Character Length Distribution')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].hist(df['char_len'].clip(upper=df['char_len'].quantile(0.99)), bins=80, color='coral', edgecolor='white')
axes[1].set_title('Character Length (99th percentile clip)')
axes[1].set_xlabel('Characters')

plt.tight_layout()
plt.show()

print(df['char_len'].describe().apply(lambda x: f'{x:,.1f}'))

In [ ]:
# ── Token-level length distribution ───────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)

# Tokenize a subsample (tokenization is slow at scale)
subsample  = df[text_col].sample(min(2000, len(df)), random_state=42).tolist()
token_lens = [len(tokenizer.encode(t, truncation=False)) for t in subsample]

plt.figure(figsize=(10, 4))
plt.hist(token_lens, bins=60, color='mediumseagreen', edgecolor='white')
plt.title('Token Length Distribution (subsample)')
plt.xlabel('Tokens')
plt.ylabel('Count')
plt.axvline(np.median(token_lens), color='red', linestyle='--', label=f'Median: {np.median(token_lens):.0f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean tokens : {np.mean(token_lens):,.1f}')
print(f'Median tokens: {np.median(token_lens):,.1f}')
print(f'Max tokens  : {max(token_lens):,}')

In [ ]:
# ── Data quality checks ────────────────────────────────────────────────────

# 1. Empty or very short samples
short   = df[df['char_len'] < 50]
print(f'Very short samples (<50 chars) : {len(short):,}  ({len(short)/len(df)*100:.2f}%)')

# 2. Duplicate detection
dupes   = df[text_col].duplicated().sum()
print(f'Exact duplicates               : {dupes:,}  ({dupes/len(df)*100:.2f}%)')

# 3. Null values
nulls   = df[text_col].isna().sum()
print(f'Null values                    : {nulls:,}')

# 4. Source distribution (if available)
if 'source' in df.columns:
    print('\nSource distribution:')
    print(df['source'].value_counts())

---
## Section 2 — Training Run

Walk through the training pipeline: load data, tokenize, run a training loop, and plot loss curves.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
PRETRAIN_LOG = Path('../pretrain/logs/train_log.jsonl')  # NeMo training log
SFT_LOG      = Path('../finetune/logs/sft_log.jsonl')
DPO_LOG      = Path('../alignment/logs/dpo_log.jsonl')

print('Log paths configured.')

In [ ]:
# ── Helper: load NeMo training log ────────────────────────────────────────
def load_log(path: Path) -> pd.DataFrame:
    """Load a jsonl training log into a DataFrame."""
    if not path.exists():
        print(f'Log not found: {path}')
        return pd.DataFrame()
    records = [json.loads(l) for l in path.read_text().splitlines() if l.strip()]
    return pd.DataFrame(records)

pretrain_df = load_log(PRETRAIN_LOG)
sft_df      = load_log(SFT_LOG)
dpo_df      = load_log(DPO_LOG)

print(f'Pretrain steps : {len(pretrain_df):,}')
print(f'SFT steps      : {len(sft_df):,}')
print(f'DPO steps      : {len(dpo_df):,}')

In [ ]:
# ── Plot pretraining loss curve ────────────────────────────────────────────
def plot_loss(df: pd.DataFrame, title: str, step_col='step', loss_col='loss', val_col='val_loss'):
    if df.empty:
        print(f'No data for: {title}')
        return
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df[step_col], df[loss_col], label='Train loss', color='steelblue', linewidth=1)
    if val_col in df.columns:
        ax.plot(df[step_col], df[val_col], label='Val loss', color='coral', linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()

plot_loss(pretrain_df, 'Pre-training Loss')
plot_loss(sft_df,      'SFT Loss')
plot_loss(dpo_df,      'DPO Loss')

In [ ]:
# ── Training throughput ────────────────────────────────────────────────────
# Tokens per second across training stages

if not pretrain_df.empty and 'tokens_per_sec' in pretrain_df.columns:
    plt.figure(figsize=(12, 3))
    plt.plot(pretrain_df['step'], pretrain_df['tokens_per_sec'], color='mediumseagreen')
    plt.title('Pre-training Throughput (tokens/sec)')
    plt.xlabel('Step')
    plt.ylabel('Tokens/sec')
    plt.tight_layout()
    plt.show()
    print(f"Mean throughput: {pretrain_df['tokens_per_sec'].mean():,.0f} tokens/sec")
else:
    print('tokens_per_sec not found in log — skipping throughput plot.')

In [ ]:
# ── Gradient norm (training stability check) ──────────────────────────────
if not pretrain_df.empty and 'grad_norm' in pretrain_df.columns:
    plt.figure(figsize=(12, 3))
    plt.plot(pretrain_df['step'], pretrain_df['grad_norm'], color='mediumpurple', linewidth=0.8)
    plt.title('Gradient Norm — Pre-training')
    plt.xlabel('Step')
    plt.ylabel('Grad norm')
    plt.tight_layout()
    plt.show()
else:
    print('grad_norm not found in log — skipping.')

---
## Section 3 — Inference

Load a trained checkpoint, generate text from prompts, and compare outputs across checkpoints.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
CHECKPOINT_BASE = Path('../pretrain/checkpoints/base')     # pretrain checkpoint
CHECKPOINT_SFT  = Path('../finetune/checkpoints/sft')      # SFT checkpoint
CHECKPOINT_DPO  = Path('../alignment/checkpoints/dpo')     # DPO checkpoint

MAX_NEW_TOKENS  = 200
TEMPERATURE     = 0.7
TOP_P           = 0.9

print('Inference config set.')

In [ ]:
# ── Load model and tokenizer ───────────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

def load_model(checkpoint_path: Path):
    """Load a HuggingFace checkpoint."""
    if not checkpoint_path.exists():
        print(f'Checkpoint not found: {checkpoint_path}')
        return None, None
    tok   = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForCausalLM.from_pretrained(checkpoint_path, torch_dtype=torch.float16)
    model = model.to(DEVICE).eval()
    print(f'Loaded: {checkpoint_path.name}  |  params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
    return model, tok

model_base, tok_base = load_model(CHECKPOINT_BASE)
model_sft,  tok_sft  = load_model(CHECKPOINT_SFT)
model_dpo,  tok_dpo  = load_model(CHECKPOINT_DPO)

In [ ]:
# ── Generate helper ────────────────────────────────────────────────────────
@torch.no_grad()
def generate(model, tokenizer, prompt: str, max_new_tokens=MAX_NEW_TOKENS,
             temperature=TEMPERATURE, top_p=TOP_P) -> str:
    if model is None:
        return '[model not loaded]'
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # Return only the newly generated tokens
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
# ── Single prompt — generate from each checkpoint ─────────────────────────
prompt = "Explain what a transformer model is in simple terms."

print(f'Prompt: {prompt}\n')
print('─' * 60)

for label, model, tok in [
    ('Base (pretrain)', model_base, tok_base),
    ('SFT',            model_sft,  tok_sft),
    ('DPO (aligned)',  model_dpo,  tok_dpo),
]:
    print(f'\n[{label}]')
    print(generate(model, tok, prompt))
    print('─' * 60)

In [ ]:
# ── Multi-prompt comparison ────────────────────────────────────────────────
prompts = [
    "What is the capital of France?",
    "Write a Python function to reverse a string.",
    "Summarise the key ideas behind reinforcement learning from human feedback.",
]

results = []
for prompt in prompts:
    results.append({
        'prompt'       : prompt,
        'base'         : generate(model_base, tok_base, prompt, max_new_tokens=100),
        'sft'          : generate(model_sft,  tok_sft,  prompt, max_new_tokens=100),
        'dpo'          : generate(model_dpo,  tok_dpo,  prompt, max_new_tokens=100),
    })

results_df = pd.DataFrame(results)

for _, row in results_df.iterrows():
    print(f'\nPrompt: {row["prompt"]}')
    print(f'  Base : {row["base"][:120]}...')
    print(f'  SFT  : {row["sft"][:120]}...')
    print(f'  DPO  : {row["dpo"][:120]}...')
    print('─' * 60)

In [ ]:
# ── Temperature sweep — effect on output diversity ─────────────────────────
# Uses the DPO model as it's the most refined checkpoint

sweep_prompt = "The future of AI is"
temperatures = [0.3, 0.7, 1.0, 1.3]

print(f'Prompt: "{sweep_prompt}"\n')
for temp in temperatures:
    output = generate(model_dpo, tok_dpo, sweep_prompt, max_new_tokens=80, temperature=temp)
    print(f'temp={temp}  →  {output[:150]}')
    print()